<a href="https://colab.research.google.com/github/luthraosu/ForecastingWIthML/blob/main/PJM_Load_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# ── Define project path ──
project_path = "/content/drive/My Drive/PJM_System_Load"
# file_path = os.path.join(project_path, "hrl_load_metered.csv")

# # ── Load only the 2 columns we need ──
# load_df = pd.read_csv(
#     file_path,
#     usecols=["datetime_beginning_ept", "mw"],   # load only needed columns
#     parse_dates=["datetime_beginning_ept"]        # parse datetime on read
# )


Mounted at /content/drive


In [ ]:
#Install dependencies:
# !pip install pandas numpy matplotlib seaborn scipy plotly

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1: IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm

### Import Weather Data

In [ ]:
# Full path to the CSV
csv_path = f"{project_path}/meteostat.csv"

# Read into a DataFrame
df_hourly_weather = pd.read_csv(csv_path)

# Quick check
print(df_hourly_weather.head())
print(df_hourly_weather.shape)

              datetime  temp  temp_f  rhum  prcp  wspd  wpgt    pres  coco
0  2026-01-01 00:00:00   2.0    35.6    54   0.0  11.0   NaN  1008.0     4
1  2026-01-01 01:00:00   2.0    35.6    54   0.0  11.0   NaN  1007.0     4
2  2026-01-01 02:00:00   1.0    33.8    59   0.0   9.0   NaN  1006.0     4
3  2026-01-01 03:00:00   1.0    33.8    59   0.0   7.0   NaN  1006.0     4
4  2026-01-01 04:00:00   1.0    33.8    61   0.0   7.0   NaN  1005.0     4
(1104, 9)


In [ ]:
df_weather = df_hourly_weather.copy()

In [ ]:
df_weather = df_weather.rename(columns={'temp': 'temp_c'})

In [ ]:
df_weather.head(1)

,datetime,temp_c,temp_f,rhum,prcp,wspd,wpgt,pres,coco
0,2026-01-01 00:00:00,2.0,35.6,54,0.0,11.0,NaN,1008.0,4


In [ ]:
print(f"Temperature range: {df_hourly_weather['temp_f'].min():.1f}°F to {df_hourly_weather['temp_f'].max():.1f}°F\n")

Temperature range: 3.2°F to 60.8°F



### Import PJM Daily Load

In [ ]:
# Full path to the CSV
csv_path1 = f"{project_path}/pjm_hourly_load.csv"

# Read into a DataFrame
df_hourly_load = pd.read_csv(csv_path1)

In [ ]:
load_df=df_hourly_load.copy()

### Merge Dataset

In [ ]:
# Study period: January 2026
START_DATE = datetime(2026, 1, 1)
END_DATE   = datetime(2026, 2, 15, 23, 0, 0)

In [ ]:
# ── Quick pre-merge check ──
print("="*60)
print("  PRE-MERGE DIAGNOSTICS")
print("="*60)
print(f"  load_df shape    : {load_df.shape}  | dtype: {load_df['datetime'].dtype}")
print(f"  weather_df shape : {df_weather.shape}  | dtype: {df_weather['datetime'].dtype}")

# Ensure both datetime columns are the same type (tz-naive)
load_df['datetime']      = pd.to_datetime(load_df['datetime']).dt.tz_localize(None)
df_weather['datetime']   = pd.to_datetime(df_weather['datetime']).dt.tz_localize(None)

# ── Merge on datetime (inner join = only hours present in both) ──
df = pd.merge(load_df, df_weather, on='datetime', how='inner')
df = df.sort_values('datetime').reset_index(drop=True)

print(f"\n  ✅ Merged shape      : {df.shape}")
print(f"  Date range         : {df['datetime'].min()}  →  {df['datetime'].max()}")
print(f"  Hours matched      : {len(df):,}  of expected 1,104")
print(f"  Unmatched (load)   : {len(load_df) - len(df)}")
print(f"  Unmatched (weather): {len(df_weather) - len(df)}")

  PRE-MERGE DIAGNOSTICS
  load_df shape    : (1104, 2)  | dtype: datetime64[ns]
  weather_df shape : (1104, 9)  | dtype: datetime64[ns]

  ✅ Merged shape      : (1104, 10)
  Date range         : 2026-01-01 00:00:00  →  2026-02-15 23:00:00
  Hours matched      : 1,104  of expected 1,104
  Unmatched (load)   : 0
  Unmatched (weather): 0


In [ ]:
# ── Add useful derived columns ──
df['hour']       = df['datetime'].dt.hour
df['day']        = df['datetime'].dt.day
df['month']      = df['datetime'].dt.month
df['day_name']   = df['datetime'].dt.day_name()
df['is_weekend'] = df['datetime'].dt.dayofweek >= 5
df['date']       = df['datetime'].dt.date

In [ ]:
# ── Wind chill (NWS formula, valid when temp_f <= 50 and wspd > 3 mph) ──
if 'wspd' in df.columns and 'temp_f' in df.columns:
    V = df['wspd'].clip(lower=3)
    T = df['temp_f']
    df['wind_chill_f'] = np.where(
        T <= 50,
        35.74 + 0.6215*T - 35.75*(V**0.16) + 0.4275*T*(V**0.16),
        T
    )
else:
    df['wind_chill_f'] = df['temp_f']

In [ ]:
# ── Missing value check ──
print("\n─── Missing Values in Merged Dataset ───")
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print("  ✅ No missing values")
else:
    for col, cnt in null_counts.items():
        print(f"  ⚠️  {col:<15} : {cnt} missing ({cnt/len(df)*100:.1f}%)")

# ── Preview ──
print("\n─── First Record ───")
print(df.iloc[[0]].to_string(index=False))

print("\n─── Last Record ───")
print(df.iloc[[-1]].to_string(index=False))

print("\n─── Columns in Merged Dataset ───")
print(list(df.columns))


─── Missing Values in Merged Dataset ───
  ⚠️  prcp            : 159 missing (14.4%)
  ⚠️  wpgt            : 894 missing (81.0%)

─── First Record ───
  datetime        mw  temp_c  temp_f  rhum  prcp  wspd  wpgt   pres  coco  hour  day  month day_name  is_weekend       date  wind_chill_f
2026-01-01 16261.788     2.0    35.6    54   0.0  11.0   NaN 1008.0     4     0    1      1 Thursday       False 2026-01-01     27.733055

─── Last Record ───
           datetime       mw  temp_c  temp_f  rhum  prcp  wspd  wpgt   pres  coco  hour  day  month day_name  is_weekend       date  wind_chill_f
2026-02-15 23:00:00 15385.04     4.0    39.2    97   0.0   6.0   NaN 1014.0     2    23   15      2   Sunday        True 2026-02-15     34.805457

─── Columns in Merged Dataset ───
['datetime', 'mw', 'temp_c', 'temp_f', 'rhum', 'prcp', 'wspd', 'wpgt', 'pres', 'coco', 'hour', 'day', 'month', 'day_name', 'is_weekend', 'date', 'wind_chill_f']


In [ ]:
# ── Note on missing data ──
# prcp : 14.4% missing  → fill with 0 (no precip recorded = dry hour)
# wpgt : 81.0% missing  → too sparse to use; EXCLUDE from analysis
# We will NOT impute wpgt — it would be misleading

df['prcp'] = df['prcp'].fillna(0)
# Drop wpgt from active analysis columns (keep in df but won't plot it)
ANALYSIS_COLS = ['mw', 'temp_f', 'wind_chill_f', 'rhum', 'prcp', 'wspd', 'pres']

print("✅ prcp NaNs filled with 0 (no precip = dry)")
print("⚠️  wpgt excluded from analysis (81% missing — too sparse to be meaningful)\n")

✅ prcp NaNs filled with 0 (no precip = dry)
⚠️  wpgt excluded from analysis (81% missing — too sparse to be meaningful)



### STATISTICAL SUMMARY

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION A: STATISTICAL SUMMARY
# ═══════════════════════════════════════════════════════════════

print("="*60)
print("  STATISTICAL SUMMARY")
print("="*60)

summary = df[ANALYSIS_COLS].describe().T
summary['skew']     = df[ANALYSIS_COLS].skew()
summary['kurtosis'] = df[ANALYSIS_COLS].kurtosis()
summary['missing']  = df[ANALYSIS_COLS].isnull().sum()

print(summary.round(2).to_string())

  STATISTICAL SUMMARY
               count      mean      std       min       25%       50%       75%       max   skew  kurtosis  missing
mw            1104.0  18265.76  2633.15  12413.92  16282.43  17872.51  20306.63  25168.38   0.27     -0.73        0
temp_f        1104.0     30.26    11.32      3.20     21.20     30.20     37.40     60.80   0.23     -0.59        0
wind_chill_f  1104.0     21.51    15.43    -10.71      8.14     22.05     31.49     60.80   0.11     -0.61        0
rhum          1104.0     55.53    19.25     19.00     41.00     51.00     69.00    100.00   0.49     -0.62        0
prcp          1104.0      0.00     0.04      0.00      0.00      0.00      0.00      0.60  11.32    148.28        0
wspd          1104.0     12.28     9.65      0.00      6.00     11.00     17.00     57.00   1.00      1.15        0
pres          1104.0   1018.52     7.20    996.00   1015.00   1019.00   1023.00   1041.00  -0.11      0.90        0


In [ ]:
# ── Pearson correlations with MW load ──
print("\n─── Pearson Correlation with MW Load ───")
for col in ['temp_f', 'wind_chill_f', 'rhum', 'prcp', 'wspd', 'pres']:
    if col in df.columns:
        valid = df[['mw', col]].dropna()
        r, p = stats.pearsonr(valid['mw'], valid[col])
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
        print(f"  {col:<16}  r = {r:+.4f}   p = {p:.2e}  {sig}")

print("\n  Significance: * p<0.05  ** p<0.01  *** p<0.001")


─── Pearson Correlation with MW Load ───
  temp_f            r = -0.7809   p = 2.07e-227  ***
  wind_chill_f      r = -0.8179   p = 7.66e-267  ***
  rhum              r = -0.3146   p = 8.83e-27  ***
  prcp              r = -0.0423   p = 1.61e-01  
  wspd              r = +0.3479   p = 9.10e-33  ***
  pres              r = +0.3650   p = 4.02e-36  ***

  Significance: * p<0.05  ** p<0.01  *** p<0.001


In [ ]:
df_jan = df[df['month'] == 1].copy()
df_feb = df[df['month'] == 2].copy()

print(f"\n  January records : {len(df_jan)}")
print(f"  February records: {len(df_feb)}")


  January records : 744
  February records: 360


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION C: PLOT 1 — Full Timeline: Load + Temperature
# ═══════════════════════════════════════════════════════════════

fig1 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("Hourly Electric Load (MW)", "Hourly Temperature & Wind Chill (°F)")
)

# Load
fig1.add_trace(go.Scatter(
    x=df['datetime'], y=df['mw'],
    name='Load (MW)', mode='lines',
    line=dict(color='#2563EB', width=1),
    hovertemplate='%{x}<br>Load: %{y:,.0f} MW<extra></extra>'
), row=1, col=1)

# Temperature
fig1.add_trace(go.Scatter(
    x=df['datetime'], y=df['temp_f'],
    name='Temp (°F)', mode='lines',
    line=dict(color='#DC2626', width=1),
    hovertemplate='%{x}<br>Temp: %{y:.1f}°F<extra></extra>'
), row=2, col=1)

# Wind chill
fig1.add_trace(go.Scatter(
    x=df['datetime'], y=df['wind_chill_f'],
    name='Wind Chill (°F)', mode='lines',
    line=dict(color='#F97316', width=1, dash='dot'),
    hovertemplate='%{x}<br>Wind Chill: %{y:.1f}°F<extra></extra>'
), row=2, col=1)

# Freezing line
fig1.add_hline(y=32, row=2, col=1,
               line=dict(color='#93C5FD', width=1, dash='dash'),
               annotation_text='32°F Freezing', annotation_position='right')

fig1.update_layout(
    title=dict(text='Electric Load & Temperature Timeline — Jan 1 to Feb 15, 2026 | IAD Area',
               font=dict(size=16)),
    height=600,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white'
)
fig1.update_yaxes(title_text='Load (MW)', row=1, col=1)
fig1.update_yaxes(title_text='Temperature (°F)', row=2, col=1)
fig1.update_xaxes(title_text='Date', row=2, col=1)

fig1.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION D: PLOT 2 — Load vs Temperature Scatter (colored by hour)
# ═══════════════════════════════════════════════════════════════

fig2 = px.scatter(
    df, x='temp_f', y='mw',
    color='hour',
    color_continuous_scale='Turbo',
    opacity=0.6,
    hover_data={'datetime': True, 'temp_f': ':.1f', 'mw': ':,.0f',
                'wind_chill_f': ':.1f', 'hour': True, 'day_name': True},
    labels={'temp_f': 'Temperature (°F)', 'mw': 'Load (MW)', 'hour': 'Hour of Day'},
    title='Load vs Temperature — Colored by Hour of Day'
)

# Add regression line
valid = df[['temp_f', 'mw']].dropna()
m, b  = np.polyfit(valid['temp_f'], valid['mw'], 1)
x_rng = np.linspace(valid['temp_f'].min(), valid['temp_f'].max(), 200)
r_val, _ = stats.pearsonr(valid['temp_f'], valid['mw'])

fig2.add_trace(go.Scatter(
    x=x_rng, y=m * x_rng + b,
    mode='lines', name=f'Regression (r={r_val:.2f})',
    line=dict(color='black', width=2, dash='dash')
))

fig2.update_layout(height=550, template='plotly_white',
                   coloraxis_colorbar=dict(title='Hour'))
fig2.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION D: PLOT 2 — Load vs Wind Chill Scatter (colored by hour)
# ═══════════════════════════════════════════════════════════════

fig2 = px.scatter(
    df, x='wind_chill_f', y='mw',
    color='hour',
    color_continuous_scale='Turbo',
    opacity=0.6,
    hover_data={'datetime': True, 'wind_chill_f': ':.1f', 'temp_f': ':.1f',
                'mw': ':,.0f', 'hour': True, 'day_name': True},
    labels={'wind_chill_f': 'Wind Chill (°F)', 'mw': 'Load (MW)', 'hour': 'Hour of Day'},
    title='Load vs Wind Chill — Colored by Hour of Day'
)

# Add regression line using wind_chill_f
valid  = df[['wind_chill_f', 'mw']].dropna()
m, b   = np.polyfit(valid['wind_chill_f'], valid['mw'], 1)
x_rng  = np.linspace(valid['wind_chill_f'].min(), valid['wind_chill_f'].max(), 200)
r_val, p_val = stats.pearsonr(valid['wind_chill_f'], valid['mw'])

fig2.add_trace(go.Scatter(
    x=x_rng, y=m * x_rng + b,
    mode='lines', name=f'Regression (r={r_val:.2f})',
    line=dict(color='black', width=2, dash='dash')
))

# Mark freezing line
fig2.add_vline(
    x=32,
    line=dict(color='#93C5FD', width=1.5, dash='dot'),
    annotation_text='32°F Freezing',
    annotation_position='top right',
    annotation_font=dict(color='#1D4ED8', size=11)
)

fig2.update_layout(
    height=550,
    template='plotly_white',
    coloraxis_colorbar=dict(title='Hour of Day'),
    annotations=[dict(
        x=0.01, y=0.97, xref='paper', yref='paper',
        text=f'<b>r = {r_val:.3f}</b>  |  p = {p_val:.2e}',
        showarrow=False, font=dict(size=12),
        bgcolor='rgba(255,255,255,0.8)', bordercolor='gray', borderwidth=1
    )]
)
fig2.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION E: PLOT 3 — Average Load Profile by Hour of Day
#            Split: Weekday vs Weekend, January vs February
# ═══════════════════════════════════════════════════════════════

hourly_profile = df.groupby(['month', 'is_weekend', 'hour'])['mw'].agg(
    mean='mean', std='std', max='max', min='min'
).reset_index()

month_map   = {1: 'January', 2: 'February'}
weekend_map = {False: 'Weekday', True: 'Weekend'}
color_map   = {
    ('January',  'Weekday'): '#1D4ED8',
    ('January',  'Weekend'): '#93C5FD',
    ('February', 'Weekday'): '#B91C1C',
    ('February', 'Weekend'): '#FCA5A5',
}

fig3 = go.Figure()
for (mo, is_wk), grp in hourly_profile.groupby(['month', 'is_weekend']):
    label = f"{month_map[mo]} – {weekend_map[is_wk]}"
    color = color_map[(month_map[mo], weekend_map[is_wk])]
    grp   = grp.sort_values('hour')
    dash  = 'solid' if not is_wk else 'dot'

    # Shaded std band
    fig3.add_trace(go.Scatter(
        x=pd.concat([grp['hour'], grp['hour'][::-1]]),
        y=pd.concat([grp['mean'] + grp['std'], (grp['mean'] - grp['std'])[::-1]]),
        fill='toself', fillcolor=color, opacity=0.12,
        line=dict(color='rgba(0,0,0,0)'), showlegend=False, hoverinfo='skip'
    ))
    # Mean line
    fig3.add_trace(go.Scatter(
        x=grp['hour'], y=grp['mean'],
        name=label, mode='lines+markers',
        line=dict(color=color, width=2, dash=dash),
        marker=dict(size=5),
        hovertemplate=f'<b>{label}</b><br>Hour: %{{x}}:00<br>Avg Load: %{{y:,.0f}} MW<extra></extra>'
    ))

fig3.update_layout(
    title='Average Hourly Load Profile — Weekday vs Weekend | Jan & Feb',
    xaxis=dict(title='Hour of Day', tickvals=list(range(0, 24)),
               ticktext=[f'{h:02d}:00' for h in range(0, 24)]),
    yaxis=dict(title='Average Load (MW)'),
    height=500, template='plotly_white', hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig3.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION F: PLOT 4 — Daily Max Load + Daily Min Temp Heatmap
# ═══════════════════════════════════════════════════════════════

daily = df.groupby('date').agg(
    max_load=('mw',     'max'),
    min_temp=('temp_f', 'min'),
    avg_temp=('temp_f', 'mean'),
    avg_load=('mw',     'mean'),
    day_name=('day_name', 'first')
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])

fig4 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('Daily Peak Load (MW)', 'Daily Min Temperature (°F)')
)

fig4.add_trace(go.Bar(
    x=daily['date'], y=daily['max_load'],
    name='Daily Peak Load',
    marker_color=daily['max_load'],
    marker_colorscale='Blues',
    hovertemplate='%{x|%b %d}<br>Peak: %{y:,.0f} MW<extra></extra>'
), row=1, col=1)

fig4.add_trace(go.Bar(
    x=daily['date'], y=daily['min_temp'],
    name='Daily Min Temp (°F)',
    marker_color=daily['min_temp'],
    marker_colorscale='RdBu_r',   # red=cold (reversed), blue=warm
    hovertemplate='%{x|%b %d}<br>Min Temp: %{y:.1f}°F<extra></extra>'
), row=2, col=1)

fig4.add_hline(y=32, row=2, col=1,
               line=dict(color='navy', width=1, dash='dash'),
               annotation_text='Freezing (32°F)')

fig4.update_layout(
    title='Daily Peak Load vs. Daily Min Temperature',
    height=580, template='plotly_white', showlegend=False
)
fig4.update_yaxes(title_text='Peak Load (MW)', row=1, col=1)
fig4.update_yaxes(title_text='Min Temp (°F)', row=2, col=1)
fig4.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION G: PLOT 5 — Box Plot: Load Distribution by Hour
# ═══════════════════════════════════════════════════════════════

fig5 = px.box(
    df, x='hour', y='mw',
    color='is_weekend',
    color_discrete_map={False: '#2563EB', True: '#F59E0B'},
    labels={'hour': 'Hour of Day', 'mw': 'Load (MW)', 'is_weekend': 'Weekend'},
    category_orders={'is_weekend': [False, True]},
    title='Load Distribution by Hour of Day (Weekday vs Weekend)',
    points=False
)
fig5.update_layout(height=500, template='plotly_white',
                   legend=dict(title='Is Weekend'))
fig5.show()


print("\n✅ All 5 plots rendered.")
print("\n─── Quick Stats Recap ───")
print(f"  Overall avg load       : {df['mw'].mean():>10,.1f} MW")
print(f"  Overall peak load      : {df['mw'].max():>10,.1f} MW  @ {df.loc[df['mw'].idxmax(), 'datetime']}")
print(f"  Overall min load       : {df['mw'].min():>10,.1f} MW  @ {df.loc[df['mw'].idxmin(), 'datetime']}")
print(f"  Coldest hour           : {df['temp_f'].min():>10.1f} °F  @ {df.loc[df['temp_f'].idxmin(), 'datetime']}")
print(f"  Warmest hour           : {df['temp_f'].max():>10.1f} °F  @ {df.loc[df['temp_f'].idxmax(), 'datetime']}")
print(f"  Coldest wind chill     : {df['wind_chill_f'].min():>10.1f} °F  @ {df.loc[df['wind_chill_f'].idxmin(), 'datetime']}")
print(f"  Temp-Load correlation  : {r_val:>+10.4f} (Pearson r)")


✅ All 5 plots rendered.

─── Quick Stats Recap ───
  Overall avg load       :   18,265.8 MW
  Overall peak load      :   25,168.4 MW  @ 2026-02-09 07:00:00
  Overall min load       :   12,413.9 MW  @ 2026-01-10 03:00:00
  Coldest hour           :        3.2 °F  @ 2026-01-27 12:00:00
  Warmest hour           :       60.8 °F  @ 2026-01-07 17:00:00
  Coldest wind chill     :      -10.7 °F  @ 2026-01-24 10:00:00
  Temp-Load correlation  :    -0.7809 (Pearson r)


### Impact of Temperature & Wind Chill on Electric Load. Levels: Correlation → Regression → Segmented → Lag Analysis

Level 1 — Correlation Comparison (you already have this)
Compare Pearson r for temp_f vs wind_chill_f against mw. Whichever has a stronger negative r is the better single predictor. But correlation alone doesn't tell you how much load changes per degree.

Level 2 — Simple Linear Regression
Quantifies the relationship: "for every 1°F drop in wind chill, load increases by X MW." Run separately for temp and wind chill, then compare the slope and R².

Level 3 — Multiple Regression
Combines both variables (and controls like hour of day, weekday/weekend) into one model. This answers: "does wind chill explain load beyond what temperature alone already explains?" If the wind chill coefficient is significant even after controlling for temperature, it's an independent driver.

Level 4 — Segmented Analysis
The relationship isn't linear across all temperatures. Load behaves differently in three zones:

Above ~55°F — load is driven by cooling or baseload, not heating
32°F–55°F — moderate heating response
Below 32°F — aggressive heating response, steepest load increase per degree


Level 5 — Lag Analysis
Load may not respond instantly to temperature. Buildings take time to cool down and thermostats to respond. Testing 1–3 hour lags on temperature against current load can reveal a delayed effect.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LEVEL 1: CORRELATION COMPARISON
# ═══════════════════════════════════════════════════════════════

print("="*60)
print("  LEVEL 1: Correlation Comparison")
print("="*60)

predictors = {
    'temp_f'       : 'Air Temperature (°F)',
    'wind_chill_f' : 'Wind Chill (°F)',
}

corr_results = {}
for col, label in predictors.items():
    valid = df[['mw', col]].dropna()
    r, p  = stats.pearsonr(valid['mw'], valid[col])
    corr_results[col] = {'r': r, 'p': p, 'label': label, 'n': len(valid)}
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else "*")
    print(f"\n  {label}")
    print(f"    Pearson r : {r:+.4f}  {sig}")
    print(f"    p-value   : {p:.2e}")
    print(f"    n records : {len(valid):,}")

winner = min(corr_results, key=lambda k: corr_results[k]['r'])
print(f"\n  → Stronger predictor: {corr_results[winner]['label']}  (r = {corr_results[winner]['r']:+.4f})")



  LEVEL 1: Correlation Comparison

  Air Temperature (°F)
    Pearson r : -0.7809  ***
    p-value   : 2.07e-227
    n records : 1,104

  Wind Chill (°F)
    Pearson r : -0.8179  ***
    p-value   : 7.66e-267
    n records : 1,104

  → Stronger predictor: Wind Chill (°F)  (r = -0.8179)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# LEVEL 2: SIMPLE LINEAR REGRESSION  (temp vs wind chill)
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  LEVEL 2: Simple Linear Regression")
print("="*60)

reg_results = {}
for col, label in predictors.items():
    valid  = df[['mw', col]].dropna()
    X      = sm.add_constant(valid[col])
    model  = sm.OLS(valid['mw'], X).fit()
    slope  = model.params[col]
    intercept = model.params['const']
    r2     = model.rsquared
    reg_results[col] = {'slope': slope, 'intercept': intercept, 'r2': r2, 'model': model}

    print(f"\n  {label}")
    print(f"    Equation  : Load = {intercept:,.1f} + ({slope:+.2f}) × {col}")
    print(f"    Slope     : {slope:+.2f} MW per °F")
    print(f"    R²        : {r2:.4f}  ({r2*100:.1f}% of load variance explained)")
    print(f"    Intercept : {intercept:,.1f} MW (theoretical load at 0°F)")

print(f"\n  → Interpretation: A 10°F drop in wind chill increases load by "
      f"~{abs(reg_results['wind_chill_f']['slope'])*10:,.0f} MW")


  LEVEL 2: Simple Linear Regression

  Air Temperature (°F)
    Equation  : Load = 23,762.7 + (-181.64) × temp_f
    Slope     : -181.64 MW per °F
    R²        : 0.6098  (61.0% of load variance explained)
    Intercept : 23,762.7 MW (theoretical load at 0°F)

  Wind Chill (°F)
    Equation  : Load = 21,267.8 + (-139.58) × wind_chill_f
    Slope     : -139.58 MW per °F
    R²        : 0.6690  (66.9% of load variance explained)
    Intercept : 21,267.8 MW (theoretical load at 0°F)

  → Interpretation: A 10°F drop in wind chill increases load by ~1,396 MW


In [ ]:
# ═══════════════════════════════════════════════════════════════
# LEVEL 3: MULTIPLE REGRESSION
# Controls for hour-of-day and weekday to isolate temp effect
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  LEVEL 3: Multiple Regression (with controls)")
print("="*60)

df_reg = df[['mw', 'temp_f', 'wind_chill_f', 'hour', 'is_weekend']].dropna().copy()
df_reg['is_weekend'] = df_reg['is_weekend'].astype(int)

# Model A: temp only + controls
XA = sm.add_constant(df_reg[['temp_f', 'hour', 'is_weekend']])
mA = sm.OLS(df_reg['mw'], XA).fit()

# Model B: wind chill only + controls
XB = sm.add_constant(df_reg[['wind_chill_f', 'hour', 'is_weekend']])
mB = sm.OLS(df_reg['mw'], XB).fit()

# Model C: both temp + wind chill + controls
XC = sm.add_constant(df_reg[['temp_f', 'wind_chill_f', 'hour', 'is_weekend']])
mC = sm.OLS(df_reg['mw'], XC).fit()

print(f"\n  {'Model':<40} {'R²':>8} {'Adj R²':>8}")
print(f"  {'-'*56}")
print(f"  {'A: Temp + Hour + Weekend':<40} {mA.rsquared:>8.4f} {mA.rsquared_adj:>8.4f}")
print(f"  {'B: WindChill + Hour + Weekend':<40} {mB.rsquared:>8.4f} {mB.rsquared_adj:>8.4f}")
print(f"  {'C: Both Temp + WindChill + Hour + Weekend':<40} {mC.rsquared:>8.4f} {mC.rsquared_adj:>8.4f}")

print(f"\n  Model C Coefficients:")
print(f"  {'Variable':<20} {'Coeff (MW)':>12} {'p-value':>12} {'Significant?':>14}")
print(f"  {'-'*60}")
for var in ['temp_f', 'wind_chill_f', 'hour', 'is_weekend']:
    coeff = mC.params.get(var, np.nan)
    pval  = mC.pvalues.get(var, np.nan)
    sig   = "Yes ***" if pval < 0.001 else ("Yes **" if pval < 0.01 else ("Yes *" if pval < 0.05 else "No"))
    print(f"  {var:<20} {coeff:>+12.2f} {pval:>12.2e} {sig:>14}")

print(f"\n  → If wind_chill_f is significant in Model C (even alongside temp_f),")
print(f"    wind chill is an INDEPENDENT driver of load — not just a proxy for temp.")



  LEVEL 3: Multiple Regression (with controls)

  Model                                          R²   Adj R²
  --------------------------------------------------------
  A: Temp + Hour + Weekend                   0.6458   0.6449
  B: WindChill + Hour + Weekend              0.6919   0.6910
  C: Both Temp + WindChill + Hour + Weekend   0.6919   0.6908

  Model C Coefficients:
  Variable               Coeff (MW)      p-value   Significant?
  ------------------------------------------------------------
  temp_f                      -4.83     7.49e-01             No
  wind_chill_f              -140.68     3.85e-35        Yes ***
  hour                       +58.73     2.95e-18        Yes ***
  is_weekend                 -41.63     6.68e-01             No

  → If wind_chill_f is significant in Model C (even alongside temp_f),
    wind chill is an INDEPENDENT driver of load — not just a proxy for temp.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# LEVEL 4: SEGMENTED ANALYSIS (temp zones)
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  LEVEL 4: Segmented Analysis by Temperature Zone")
print("="*60)

bins   = [-999, 20, 32, 45, 55, 999]
labels = ['<20°F (Extreme Cold)', '20–32°F (Below Freezing)',
          '32–45°F (Cold)', '45–55°F (Mild)', '>55°F (Warm)']

df['temp_zone'] = pd.cut(df['temp_f'], bins=bins, labels=labels)

seg = df.groupby('temp_zone', observed=True).agg(
    hours     = ('mw', 'count'),
    avg_load  = ('mw', 'mean'),
    max_load  = ('mw', 'max'),
    avg_temp  = ('temp_f', 'mean'),
    avg_wchill= ('wind_chill_f', 'mean'),
).reset_index()

# Slope per zone
print(f"\n  {'Zone':<26} {'Hours':>6} {'Avg Load':>10} {'Max Load':>10} {'Avg Temp':>10} {'Load/°F Slope':>14}")
print(f"  {'-'*80}")
for _, row in seg.iterrows():
    zone_data = df[df['temp_zone'] == row['temp_zone']][['temp_f', 'mw']].dropna()
    slope_str = 'n/a'
    if len(zone_data) > 5:
        m_z, _ = np.polyfit(zone_data['temp_f'], zone_data['mw'], 1)
        slope_str = f"{m_z:+.1f} MW/°F"
    print(f"  {str(row['temp_zone']):<26} {int(row['hours']):>6} "
          f"{row['avg_load']:>10,.0f} {row['max_load']:>10,.0f} "
          f"{row['avg_temp']:>9.1f}° {slope_str:>14}")

print(f"\n  → The steepest MW/°F slope = strongest heating response zone")




  LEVEL 4: Segmented Analysis by Temperature Zone

  Zone                        Hours   Avg Load   Max Load   Avg Temp  Load/°F Slope
  --------------------------------------------------------------------------------
  <20°F (Extreme Cold)          255     21,442     25,168      15.7°    -68.6 MW/°F
  20–32°F (Below Freezing)      383     18,607     23,247      26.8°   -227.5 MW/°F
  32–45°F (Cold)                338     16,647     20,791      37.7°    -78.0 MW/°F
  45–55°F (Mild)                111     15,124     17,787      49.0°    -30.5 MW/°F
  >55°F (Warm)                   17     15,623     16,628      57.1°    -95.1 MW/°F

  → The steepest MW/°F slope = strongest heating response zone


In [ ]:
# ═══════════════════════════════════════════════════════════════
# LEVEL 5: LAG ANALYSIS
# Does load respond to temperature with a 1–3 hour delay?
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  LEVEL 5: Lag Analysis (Delayed Temperature Effect)")
print("="*60)

df_lag = df[['datetime', 'mw', 'wind_chill_f']].sort_values('datetime').copy()
print(f"\n  {'Lag (hours)':<14} {'Pearson r':>12} {'p-value':>14} {'Interpretation'}")
print(f"  {'-'*70}")

lag_correlations = {}
for lag in range(0, 5):
    df_lag[f'wc_lag{lag}'] = df_lag['wind_chill_f'].shift(lag)
    valid = df_lag[['mw', f'wc_lag{lag}']].dropna()
    r, p  = stats.pearsonr(valid['mw'], valid[f'wc_lag{lag}'])
    lag_correlations[lag] = r
    note = "(current hour)" if lag == 0 else f"(temp {lag}h ago → current load)"
    print(f"  Lag {lag:<10} {r:>+12.4f} {p:>14.2e}   {note}")

best_lag = max(lag_correlations, key=lambda k: abs(lag_correlations[k]))
print(f"\n  → Strongest correlation at lag = {best_lag} hour(s)  "
      f"(r = {lag_correlations[best_lag]:+.4f})")
if best_lag > 0:
    print(f"    Load responds to wind chill with a ~{best_lag}h delay — "
          f"buildings take time to lose heat.")
else:
    print(f"    Load responds to wind chill in the same hour — near-instantaneous response.")



  LEVEL 5: Lag Analysis (Delayed Temperature Effect)

  Lag (hours)       Pearson r        p-value Interpretation
  ----------------------------------------------------------------------
  Lag 0               -0.8179      7.66e-267   (current hour)
  Lag 1               -0.8038      1.29e-250   (temp 1h ago → current load)
  Lag 2               -0.7932      3.35e-239   (temp 2h ago → current load)
  Lag 3               -0.7869      1.24e-232   (temp 3h ago → current load)
  Lag 4               -0.7849      1.82e-230   (temp 4h ago → current load)

  → Strongest correlation at lag = 0 hour(s)  (r = -0.8179)
    Load responds to wind chill in the same hour — near-instantaneous response.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PLOTS: Regression lines comparison + Segmented scatter
# ═══════════════════════════════════════════════════════════════

# ── Plot A: Overlay regression lines for temp vs wind chill ──
fig_A = go.Figure()

for col, label, color in [
    ('temp_f',       'Air Temp (°F)',   '#DC2626'),
    ('wind_chill_f', 'Wind Chill (°F)', '#F97316'),
]:
    valid = df[[col, 'mw']].dropna()
    fig_A.add_trace(go.Scatter(
        x=valid[col], y=valid['mw'],
        mode='markers', name=f'{label} (data)',
        marker=dict(color=color, opacity=0.3, size=4),
        hovertemplate=f'{label}: %{{x:.1f}}°F<br>Load: %{{y:,.0f}} MW<extra></extra>'
    ))
    m_l, b_l = np.polyfit(valid[col], valid['mw'], 1)
    x_rng    = np.linspace(valid[col].min(), valid[col].max(), 200)
    r_l, _   = stats.pearsonr(valid[col], valid['mw'])
    fig_A.add_trace(go.Scatter(
        x=x_rng, y=m_l * x_rng + b_l,
        mode='lines', name=f'{label} regression (r={r_l:.3f})',
        line=dict(color=color, width=2.5)
    ))

fig_A.add_vline(x=32, line=dict(color='navy', width=1, dash='dot'),
                annotation_text='32°F', annotation_position='top right')
fig_A.update_layout(
    title='Load vs Temperature & Wind Chill — Regression Comparison',
    xaxis_title='Temperature / Wind Chill (°F)',
    yaxis_title='Load (MW)',
    height=500, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1)
)
fig_A.show()


# ── Plot B: Segmented box plots (load by temp zone) ──
fig_B = px.box(
    df.dropna(subset=['temp_zone']),
    x='temp_zone', y='mw',
    color='temp_zone',
    color_discrete_sequence=px.colors.sequential.RdBu_r,
    labels={'temp_zone': 'Temperature Zone', 'mw': 'Load (MW)'},
    title='Load Distribution by Temperature Zone',
    points=False,
    category_orders={'temp_zone': labels}
)
fig_B.update_layout(height=480, template='plotly_white', showlegend=False)
fig_B.show()


# ── Plot C: Lag correlation bar chart ──
fig_C = go.Figure(go.Bar(
    x=[f'Lag {k}h' for k in lag_correlations],
    y=list(lag_correlations.values()),
    marker_color=['#F59E0B' if k == best_lag else '#2563EB'
                  for k in lag_correlations],
    text=[f'r={v:+.3f}' for v in lag_correlations.values()],
    textposition='outside'
))
fig_C.update_layout(
    title='Lag Analysis — Wind Chill vs Load Correlation by Hour Offset',
    xaxis_title='Lag (hours)',
    yaxis_title='Pearson r',
    height=400, template='plotly_white',
    yaxis=dict(range=[min(lag_correlations.values()) - 0.05,
                      max(lag_correlations.values()) + 0.05])
)
fig_C.show()


### Spike Even Identification

In [ ]:
SPIKE_THRESHOLD = 23500  # MW

# ═══════════════════════════════════════════════════════════════
# SECTION A: IDENTIFY SPIKE HOURS & GROUP INTO EVENTS
# Consecutive hours above threshold = one event
# ═══════════════════════════════════════════════════════════════

print("="*60)
print(f"  SPIKE ANALYSIS  |  Threshold: {SPIKE_THRESHOLD:,} MW")
print("="*60)

df_sorted = df.sort_values('datetime').copy()

# Flag spike hours
df_sorted['is_spike'] = df_sorted['mw'] >= SPIKE_THRESHOLD

# Group consecutive spike hours into events using cumsum trick
df_sorted['event_id'] = (
    df_sorted['is_spike'] != df_sorted['is_spike'].shift()
).cumsum()
df_sorted.loc[~df_sorted['is_spike'], 'event_id'] = np.nan

# Spike hours only
spikes = df_sorted[df_sorted['is_spike']].copy()

print(f"\n  Total spike hours (≥{SPIKE_THRESHOLD:,} MW) : {len(spikes)}")
print(f"  Max load recorded                  : {spikes['mw'].max():,.1f} MW  @ {spikes.loc[spikes['mw'].idxmax(), 'datetime']}")
print(f"  Min load during spikes             : {spikes['mw'].min():,.1f} MW")
print(f"  Avg load during spikes             : {spikes['mw'].mean():,.1f} MW")

# Summarise each event
events = spikes.groupby('event_id').agg(
    start         = ('datetime', 'min'),
    end           = ('datetime', 'max'),
    duration_hrs  = ('datetime', 'count'),
    peak_load     = ('mw', 'max'),
    avg_load      = ('mw', 'mean'),
    min_temp_f    = ('temp_f', 'min'),
    avg_temp_f    = ('temp_f', 'mean'),
    min_wc_f      = ('wind_chill_f', 'min'),
    avg_wc_f      = ('wind_chill_f', 'mean'),
    avg_wspd      = ('wspd', 'mean'),
    avg_rhum      = ('rhum', 'mean'),
).reset_index(drop=True)

events.index = events.index + 1  # event number starts at 1

print(f"\n  Number of distinct spike events    : {len(events)}")
print(f"\n─── Event Summary ───")
print(f"\n  {'#':<4} {'Start':<22} {'End':<22} {'Hrs':>4} {'Peak MW':>10} "
      f"{'Min Temp':>10} {'Min WC':>8} {'Avg Wind':>10}")
print(f"  {'-'*94}")
for i, row in events.iterrows():
    print(f"  {i:<4} {str(row['start']):<22} {str(row['end']):<22} "
          f"{int(row['duration_hrs']):>4} {row['peak_load']:>10,.1f} "
          f"{row['min_temp_f']:>9.1f}°F {row['min_wc_f']:>7.1f}°F "
          f"{row['avg_wspd']:>9.1f} mph")



  SPIKE ANALYSIS  |  Threshold: 23,500 MW

  Total spike hours (≥23,500 MW) : 29
  Max load recorded                  : 25,168.4 MW  @ 2026-02-09 07:00:00
  Min load during spikes             : 23,501.3 MW
  Avg load during spikes             : 23,864.3 MW

  Number of distinct spike events    : 10

─── Event Summary ───

  #    Start                  End                     Hrs    Peak MW   Min Temp   Min WC   Avg Wind
  ----------------------------------------------------------------------------------------------
  1    2026-01-21 07:00:00    2026-01-21 07:00:00       1   23,860.2      10.4°F     4.9°F       0.0 mph
  2    2026-01-27 06:00:00    2026-01-27 08:00:00       3   24,104.6      12.2°F    -2.0°F      13.0 mph
  3    2026-01-29 06:00:00    2026-01-29 08:00:00       3   24,196.4      15.8°F    -1.3°F      18.7 mph
  4    2026-01-30 06:00:00    2026-01-30 08:00:00       3   24,162.3      10.4°F     0.8°F       5.7 mph
  5    2026-01-31 07:00:00    2026-01-31 09:00:00       3  

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION B: PRE-SPIKE TEMPERATURE RUNWAY
# Were temps continuously falling before each spike?
# Look at 24 hours before each event start
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  SECTION B: Pre-Spike Temperature Runway (24h Before Event)")
print("="*60)

runway_hours = 24

for i, row in events.iterrows():
    window_start = row['start'] - pd.Timedelta(hours=runway_hours)
    window_end   = row['start'] - pd.Timedelta(hours=1)
    pre = df_sorted[(df_sorted['datetime'] >= window_start) &
                    (df_sorted['datetime'] <= window_end)]

    if len(pre) == 0:
        print(f"\n  Event {i}: insufficient pre-event data")
        continue

    temp_drop = pre['temp_f'].iloc[0] - pre['temp_f'].iloc[-1]
    wc_drop   = pre['wind_chill_f'].iloc[0] - pre['wind_chill_f'].iloc[-1]
    hrs_below_freezing = (pre['temp_f'] < 32).sum()

    print(f"\n  Event {i} — starts {row['start']}")
    print(f"    Temp 24h before start : {pre['temp_f'].iloc[0]:.1f}°F")
    print(f"    Temp at event start   : {pre['temp_f'].iloc[-1]:.1f}°F")
    print(f"    Temperature drop      : {temp_drop:+.1f}°F over {runway_hours}h "
          f"({'falling' if temp_drop > 0 else 'rising'})")
    print(f"    Wind chill drop       : {wc_drop:+.1f}°F over {runway_hours}h")
    print(f"    Hours below freezing  : {hrs_below_freezing} of {len(pre)}")




  SECTION B: Pre-Spike Temperature Runway (24h Before Event)

  Event 1 — starts 2026-01-21 07:00:00
    Temp 24h before start : 21.2°F
    Temp at event start   : 10.9°F
    Temperature drop      : +10.3°F over 24h (falling)
    Wind chill drop       : +0.7°F over 24h
    Hours below freezing  : 24 of 24

  Event 2 — starts 2026-01-27 06:00:00
    Temp 24h before start : 15.8°F
    Temp at event start   : 17.6°F
    Temperature drop      : -1.8°F over 24h (rising)
    Wind chill drop       : +2.8°F over 24h
    Hours below freezing  : 24 of 24

  Event 3 — starts 2026-01-29 06:00:00
    Temp 24h before start : 19.4°F
    Temp at event start   : 17.6°F
    Temperature drop      : +1.8°F over 24h (falling)
    Wind chill drop       : -2.7°F over 24h
    Hours below freezing  : 24 of 24

  Event 4 — starts 2026-01-30 06:00:00
    Temp 24h before start : 17.6°F
    Temp at event start   : 14.0°F
    Temperature drop      : +3.6°F over 24h (falling)
    Wind chill drop       : +3.7°F over

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION C: WHAT OTHER VARIABLES EXPLAIN THE SPIKES?
# Compare spike hours vs all other hours across all variables
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  SECTION C: Variable Comparison — Spike vs Non-Spike Hours")
print("="*60)

non_spikes = df_sorted[~df_sorted['is_spike']]

compare_vars = {
    'temp_f'       : 'Air Temp (°F)',
    'wind_chill_f' : 'Wind Chill (°F)',
    'wspd'         : 'Wind Speed (mph)',
    'rhum'         : 'Rel Humidity (%)',
    'prcp'         : 'Precipitation (mm)',
    'pres'         : 'Pressure (hPa)',
    'hour'         : 'Hour of Day',
}

print(f"\n  {'Variable':<22} {'Spike Avg':>12} {'Non-Spike Avg':>14} "
      f"{'Difference':>12} {'t-stat':>9} {'p-value':>12} {'Significant?':>13}")
print(f"  {'-'*100}")

ttest_results = {}
for col, label in compare_vars.items():
    if col not in df_sorted.columns:
        continue
    s_vals  = spikes[col].dropna()
    ns_vals = non_spikes[col].dropna()
    if len(s_vals) < 3 or len(ns_vals) < 3:
        continue
    t, p = stats.ttest_ind(s_vals, ns_vals)
    diff  = s_vals.mean() - ns_vals.mean()
    sig   = "Yes ***" if p < 0.001 else ("Yes **" if p < 0.01 else ("Yes *" if p < 0.05 else "No"))
    ttest_results[col] = {'label': label, 'spike_avg': s_vals.mean(),
                          'nonspike_avg': ns_vals.mean(), 'diff': diff, 'p': p}
    print(f"  {label:<22} {s_vals.mean():>12.2f} {ns_vals.mean():>14.2f} "
          f"{diff:>+12.2f} {t:>9.2f} {p:>12.2e} {sig:>13}")

print("\n  Significance: * p<0.05  ** p<0.01  *** p<0.001")
print("  Significant difference = that variable behaves differently during spikes")



  SECTION C: Variable Comparison — Spike vs Non-Spike Hours

  Variable                  Spike Avg  Non-Spike Avg   Difference    t-stat      p-value  Significant?
  ----------------------------------------------------------------------------------------------------
  Air Temp (°F)                 14.87          30.68       -15.81     -7.61     5.84e-14       Yes ***
  Wind Chill (°F)               -0.18          22.09       -22.27     -7.88     7.85e-15       Yes ***
  Wind Speed (mph)              16.45          12.17        +4.28      2.36     1.84e-02         Yes *
  Rel Humidity (%)              52.28          55.61        -3.34     -0.92     3.57e-01            No
  Precipitation (mm)             0.00           0.00        -0.00     -0.68     4.98e-01            No
  Pressure (hPa)              1022.72        1018.41        +4.32      3.20     1.43e-03        Yes **
  Hour of Day                    8.69          11.58        -2.89     -2.22     2.67e-02         Yes *

  Signific

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION D: HOUR-OF-DAY PATTERN OF SPIKES
# Are spikes concentrated at certain hours?
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  SECTION D: Hour-of-Day Distribution of Spike Hours")
print("="*60)

hour_counts = spikes['hour'].value_counts().sort_index()
print(f"\n  {'Hour':<8} {'Spike Hours':>12}  {'Bar'}")
print(f"  {'-'*40}")
for h, cnt in hour_counts.items():
    bar = "█" * cnt
    print(f"  {h:02d}:00   {cnt:>12}  {bar}")



  SECTION D: Hour-of-Day Distribution of Spike Hours

  Hour      Spike Hours  Bar
  ----------------------------------------
  05:00              1  █
  06:00              6  ██████
  07:00              9  █████████
  08:00              7  ███████
  09:00              2  ██
  17:00              1  █
  18:00              1  █
  19:00              1  █
  20:00              1  █


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION E: CONTINUOUS COLD EXPOSURE METRIC
# HDD = Heating Degree Days (daily sum of degrees below 65°F base)
# Cumulative HDD leading up to spike = cold stress accumulation
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  SECTION E: Heating Degree Days (HDD) — Cold Stress Metric")
print("="*60)

BASE_TEMP = 65  # °F standard HDD base

df_sorted['hdd_hour'] = (BASE_TEMP - df_sorted['temp_f']).clip(lower=0) / 24
daily_hdd = df_sorted.groupby('date')['hdd_hour'].sum().reset_index()
daily_hdd.columns = ['date', 'hdd']
daily_hdd['cum_hdd_3d'] = daily_hdd['hdd'].rolling(3).sum()   # 3-day rolling HDD
daily_hdd['cum_hdd_7d'] = daily_hdd['hdd'].rolling(7).sum()   # 7-day rolling HDD

print(f"\n  Daily HDD (base 65°F) — highest 10 days:")
print(daily_hdd.nlargest(10, 'hdd')[['date', 'hdd', 'cum_hdd_3d', 'cum_hdd_7d']].to_string(index=False))
print(f"\n  → Days with high 3-day or 7-day cumulative HDD = sustained cold stress")
print(f"    These days are most likely to show load spikes")



  SECTION E: Heating Degree Days (HDD) — Cold Stress Metric

  Daily HDD (base 65°F) — highest 10 days:
      date    hdd  cum_hdd_3d  cum_hdd_7d
2026-01-31 49.950    146.1000    331.5000
2026-01-24 49.350    100.3500    249.8325
2026-01-25 49.350    127.4250    268.2825
2026-01-30 49.275    140.7750    330.9000
2026-02-08 47.400    132.7425    280.6350
2026-01-27 46.950    140.7750    282.9525
2026-01-29 46.875    138.4500    310.3500
2026-02-01 46.575    145.8000    328.7250
2026-01-28 44.625    136.0500    285.7500
2026-01-26 44.475    143.1750    275.9775

  → Days with high 3-day or 7-day cumulative HDD = sustained cold stress
    These days are most likely to show load spikes


### Extra

In [ ]:
# Study period: January 2026
START_DATE = datetime(2026, 1, 1)
END_DATE   = datetime(2026, 2, 15, 23, 0, 0)

In [ ]:
 # Create hourly datetime index for January 2026
hours = pd.date_range(start=START_DATE, end=END_DATE, freq='h')

In [ ]:
n = len(hours)

In [ ]:
np.random.seed(42)

In [ ]:
# Base load ~1500 MW
base = 22000

In [ ]:
# Hour-of-day pattern (MW above/below base)
hour_of_day = np.array(hours.hour)
daily_cycle = (
-150 * np.cos(2 * np.pi * (hour_of_day - 17) / 24)   # peak ~5-8pm
+ 50  * np.sin(2 * np.pi * (hour_of_day - 8)  / 24)   # secondary morning peak
)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 5: MERGE & ANALYZE
# ─────────────────────────────────────────────

def merge_and_analyze(load_df, weather_df):
    """Merge load and weather data, compute derived features, identify peaks."""
    print("="*60)
    print("STEP 3: Merging Datasets & Computing Features")
    print("="*60)

    # Merge on datetime
    df = pd.merge(load_df, weather_df, on='datetime', how='inner')
    df = df.sort_values('datetime').reset_index(drop=True)

    # Derived time features
    df['hour']        = df['datetime'].dt.hour
    df['day']         = df['datetime'].dt.day
    df['dayofweek']   = df['datetime'].dt.dayofweek
    df['day_name']    = df['datetime'].dt.day_name()
    df['is_weekend']  = df['dayofweek'] >= 5
    df['date_label']  = df['datetime'].dt.strftime('%b %d, %Y %I:%M %p')

    # Wind chill (for below-50°F temps)
    # NWS formula: WC = 35.74 + 0.6215T - 35.75(V^0.16) + 0.4275T(V^0.16)
    if 'wspd' in df.columns:
        V = df['wspd'].clip(lower=3)  # formula valid for V > 3 mph
        T = df['temp_f']
        df['wind_chill_f'] = np.where(
            T <= 50,
            35.74 + 0.6215*T - 35.75*(V**0.16) + 0.4275*T*(V**0.16),
            T
        )
    else:
        df['wind_chill_f'] = df['temp_f']

    # Daily stats for comparison
    daily = df.groupby('day').agg(
        daily_max_load=('load_mw', 'max'),
        daily_min_load=('load_mw', 'min'),
        daily_avg_load=('load_mw', 'mean'),
        daily_min_temp_f=('temp_f', 'min'),
        daily_avg_temp_f=('temp_f', 'mean'),
    ).reset_index()
    df = df.merge(daily, on='day', how='left')

    # Load deviation from daily average (highlights spikes)
    df['load_vs_daily_avg'] = df['load_mw'] - df['daily_avg_load']

    print(f"  Merged dataset: {len(df)} hourly records\n")
    return df

In [ ]:
# Step 1: Load electric data
load_df = load_electric_data()

# Step 2: Fetch weather data
weather_df = load_weather_data()
# Step 3: Merge and derive features
df = merge_and_analyze(load_df, weather_df)